# Agentic Workflow: Orchestrator-Workers Pattern (Dynamic Roles)

This notebook demonstrates the **Orchestrator-Workers** workflow with **Dynamic Roles (Orchestrator Decides at Runtime)**. 

In this design:
1. A central **Orchestrator** agent receives a complex prompt (specifically, a custom personal investment query involving arbitrary asset classes).
2. The Orchestrator dynamically plans and designs 2-3 specialized **Worker** agent personas (names, system instructions, and target prompts) tailored specifically to the query, outputting the plan in structured JSON.
3. The Orchestrator instantiates and dispatches these dynamic workers in parallel using python threads.
4. Finally, the Orchestrator synthesizes the worker reports into a single, cohesive wealth strategy proposal.

## Pattern Overview

```
                  ┌──────────────────────┐
                  │     ORCHESTRATOR     │
                  │  (Dynamically Plans  │
                  │   Worker Roles in    │
                  │    Structured JSON)  │
                  └──────────┬───────────┘
                             │
         ┌───────────────────┼───────────────────┐
         ▼                   ▼                   ▼
┌─────────────────┐ ┌─────────────────┐ ┌─────────────────┐
│ DYNAMIC WORKER  │ │ DYNAMIC WORKER  │ │ DYNAMIC WORKER  │
│    Persona A    │ │    Persona B    │ │    Persona C    │
│  (e.g., Crypto) │ │ (e.g., Gold)    │ │  (e.g., Bonds)  │
└────────┬────────┘ └────────┬────────┘ └────────┬────────┘
         │                   │                   │
         └───────────────────┼───────────────────┘
                             ▼
                  ┌──────────────────────┐
                  │     ORCHESTRATOR     │
                  │   (Synthesizes)      │
                  └──────────────────────┘
```

## 1. Setup and Client Initialization

We load the environment variables from the `.env` file in the parent directory and initialize the OpenAI client pointing to the Gemini API's OpenAI-compatible endpoint.

In [1]:
import os
import json
import re
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env in the parent directory
dir_path = os.path.dirname(__file__) if "__file__" in globals() else "."
dotenv_path = os.path.abspath(os.path.join(dir_path, "..", ".env"))
load_dotenv(dotenv_path=dotenv_path)

# Retrieve the API key from environment variables
api_key = os.getenv("GEMINI_API_KEY")

if not api_key or api_key == "your_actual_api_key_here":
    raise ValueError(
        "Please set your GEMINI_API_KEY in the .env file in the root of the project.\n"
        "You can get a free key from Google AI Studio: https://aistudio.google.com/"
    )

# Initialize the OpenAI client pointing to Gemini's OpenAI-compatible API endpoint
client = OpenAI(
    api_key=api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# We use the free gemini-3.1-flash-lite model via the OpenAI compatible endpoint.
model_name = "gemini-3.1-flash-lite"
print("Gemini API client initialized successfully!")

Gemini API client initialized successfully!


## 2. Base Completion Helper & JSON Cleaning

We define a helper function to make completions to Gemini, and a robust parser to strip markdown code blocks and extract raw JSON strings from the role planner's output.

In [2]:
def get_completion(prompt, system_instruction="You are a helpful assistant."):
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": prompt}
            ],
            temperature=0.2,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

def clean_json_string(raw_str):
    # Remove markdown code block wrappers (e.g. ```json ... ```) if present
    cleaned = re.sub(r"^```json\s*", "", raw_str, flags=re.IGNORECASE)
    cleaned = re.sub(r"^```\s*", "", cleaned)
    cleaned = re.sub(r"```$", "", cleaned)
    return cleaned.strip()

## 3. Dynamic Planning & Orchestration Logic

The orchestrator executes in three phases:
1. **Plan Worker Personas**: Analyzes the user's investment goals and assets to plan 2-3 customized specialists, returning their definitions in structured JSON.
2. **Execute Worker Agents**: Instantiates generic agents with the custom system instructions and dispatches them in parallel.
3. **Synthesize Final Strategy**: Gathers the specialist reports and synthesizes them into a single, cohesive wealth strategy proposal.

In [3]:
ROLE_PLANNER_SYSTEM_INSTRUCTION = (
    "You are an Investment Strategy Director. Your job is to analyze the user's investment query "
    "and dynamically determine the specialized worker roles needed to address the query. "
    "You should define exactly 2 to 3 specialized worker agents based on the specific asset classes or topics mentioned. "
    "For each agent, provide:\n"
    "1. name: A short name/role description (e.g., 'Bonds Specialist', 'Crypto Analyst').\n"
    "2. system_instruction: Detailed instructions for the agent detailing their persona, what to focus on, and what to ignore. "
    "Instruct them to analyze the query in the context of their specific domain.\n"
    "3. target_prompt: A specific query or task statement for this specialist based on the user's profile.\n\n"
    "You MUST respond ONLY with a valid JSON object matching the following structure:\n"
    "{\n"
    "  \"workers\": [\n"
    "    {\n"
    "      \"name\": \"Agent Name\",\n"
    "      \"system_instruction\": \"Persona and guidelines...\",\n"
    "      \"target_prompt\": \"Specific analysis prompt...\"\n"
    "    }\n"
    "  ]\n"
    "}"
)

ORCHESTRATOR_SYSTEM_INSTRUCTION = (
    "You are a Principal Wealth Architect. Your role is to coordinate specialist financial advisors "
    "and synthesize their findings into a cohesive, executive-level personal investment plan. "
    "You will receive a user query and several specialist reports.\n\n"
    "Synthesize these documents into a single, high-quality, professional proposal. "
    "The synthesized response MUST contain:\n"
    "- An Executive Summary of the user's situation and goals.\n"
    "- A proposed Asset Allocation tailored to their query, risk profile, and capital.\n"
    "- A summary of the key advantages and risks of each approach as highlighted by the specialists.\n"
    "- Tax Strategy and Implementation Highlights.\n"
    "- A step-by-step Action Plan/Next Steps.\n\n"
    "Ensure the final output is unified, does not repeat the exact text of the workers, resolves any conflicting advice, "
    "and reads like it was written by a single expert advisor. Use professional styling and markdown formatting."
)

def run_dynamic_worker(name, system_instruction, prompt):
    print(f"[{name}] Starting analysis...")
    report = get_completion(prompt, system_instruction=system_instruction)
    print(f"[{name}] Analysis complete!")
    return report

def run_orchestrator_workflow(user_query):
    print("Step 1: Analyzing query and dynamically planning worker roles...")
    raw_plan = get_completion(
        f"User Query: {user_query}", 
        system_instruction=ROLE_PLANNER_SYSTEM_INSTRUCTION
    )
    
    # Parse the planned JSON
    try:
        cleaned_plan = clean_json_string(raw_plan)
        plan_data = json.loads(cleaned_plan)
        workers_info = plan_data.get("workers", [])
    except Exception as e:
        print(f"Error parsing dynamic roles plan: {e}")
        print(f"Raw plan was:\n{raw_plan}")
        return None
        
    print(f"\nOrchestrator planned {len(workers_info)} specialized workers:")
    for idx, w in enumerate(workers_info):
        print(f"  {idx+1}. {w['name']}")
    print()
    
    print("Step 2: Dispatching dynamic workers in parallel...")
    reports = {}
    with ThreadPoolExecutor(max_workers=len(workers_info)) as executor:
        futures = {
            executor.submit(
                run_dynamic_worker, 
                w["name"], 
                w["system_instruction"], 
                w["target_prompt"]
            ): w["name"]
            for w in workers_info
        }
        
        for future in futures:
            name = futures[future]
            reports[name] = future.result()
            
    print("\nWorkers have completed their analysis!")
    
    # Print individual worker reports for visibility (previews)
    for name, report in reports.items():
        print("-" * 50)
        print(f"--- {name.upper()} REPORT (PREVIEW) ---")
        if report:
            print("\n".join(report.split("\n")[:5]) + "\n...")
        else:
            print("Failed to get report.")
    print("-" * 50)
    
    print("Step 3: Synthesizing specialized reports into final proposal...")
    
    reports_text = ""
    for idx, (name, report) in enumerate(reports.items()):
        reports_text += f"--- Specialist Report {idx+1}: {name} ---\n{report}\n\n"
        
    synthesis_prompt = (
        f"User Query: {user_query}\n\n"
        f"{reports_text}"
    )
    
    final_proposal = get_completion(synthesis_prompt, system_instruction=ORCHESTRATOR_SYSTEM_INSTRUCTION)
    return final_proposal

## 4. Execution Demos

We run the pipeline with two different queries to demonstrate how the Orchestrator dynamically designs different specialist agents based on the inputs.

In [4]:
print("=" * 80)
print("TEST CASE 1: CRYPTOCURRENCY, GOLD, AND BONDS")
print("=" * 80)

query_1 = (
    "I am 45 years old, looking to allocate $100,000. I am very conservative but "
    "want to put 10% in high-risk crypto. The rest should be in government bonds "
    "and gold as inflation hedges. How should I proceed?"
)

proposal_1 = run_orchestrator_workflow(query_1)
print("\n=== FINAL PROPOSAL (TEST CASE 1) ===\n")
print(proposal_1)

TEST CASE 1: CRYPTOCURRENCY, GOLD, AND BONDS
Step 1: Analyzing query and dynamically planning worker roles...

Orchestrator planned 3 specialized workers:
  1. Fixed Income & Macro Strategist
  2. Digital Asset Risk Analyst
  3. Portfolio Construction Architect

Step 2: Dispatching dynamic workers in parallel...
[Fixed Income & Macro Strategist] Starting analysis...
[Digital Asset Risk Analyst] Starting analysis...
[Portfolio Construction Architect] Starting analysis...
[Portfolio Construction Architect] Analysis complete!
[Digital Asset Risk Analyst] Analysis complete!
[Fixed Income & Macro Strategist] Analysis complete!

Workers have completed their analysis!
--------------------------------------------------
--- FIXED INCOME & MACRO STRATEGIST REPORT (PREVIEW) ---
For a 45-year-old conservative investor, the primary objective is the preservation of purchasing power. In an environment characterized by fiscal uncertainty and persistent inflationary pressures, a portfolio consisting ex

In [5]:
print("=" * 80)
print("TEST CASE 2: REAL ESTATE SYNDICATIONS AND DIVIDEND STOCKS")
print("=" * 80)

query_2 = (
    "I'm a 28-year-old software engineer in a high tax bracket. I have $150,000 to invest "
    "and want high passive income. I'm looking at real estate syndications and high-yielding "
    "dividend stocks. What is the best strategy and tax setup?"
)

proposal_2 = run_orchestrator_workflow(query_2)
print("\n=== FINAL PROPOSAL (TEST CASE 2) ===\n")
print(proposal_2)

TEST CASE 2: REAL ESTATE SYNDICATIONS AND DIVIDEND STOCKS
Step 1: Analyzing query and dynamically planning worker roles...

Orchestrator planned 2 specialized workers:
  1. Real Estate Syndication Strategist
  2. Dividend Equity & Tax Optimization Specialist

Step 2: Dispatching dynamic workers in parallel...
[Real Estate Syndication Strategist] Starting analysis...
[Dividend Equity & Tax Optimization Specialist] Starting analysis...
[Real Estate Syndication Strategist] Analysis complete!
[Dividend Equity & Tax Optimization Specialist] Analysis complete!

Workers have completed their analysis!
--------------------------------------------------
--- REAL ESTATE SYNDICATION STRATEGIST REPORT (PREVIEW) ---
For a 28-year-old high-earner, private real estate syndications represent a powerful vehicle for wealth compounding, provided the investor understands the trade-off between tax efficiency and liquidity. At this stage of your career, your primary advantage is your time horizon; your prima